In [1]:
import os

for name in os.listdir("/kaggle/input/datasets/zlemglsmklkaya"):
    print(name)

caai-models-nb3-complete
caai-models-nb1-complete
caai-models-cic-scenario-t
caai-revision-splits-new
caai-shap-nb5-complete
caai-ig-nb4-complete
caai-revision-splits


In [2]:
import os, json
import numpy as np
import pandas as pd

SPLITS_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
NB1_ROOT    = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb1-complete"
OLD_CC_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits-new"
AE_T_ROOT   = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-cic-scenario-t"
NB3_ROOT    = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb3-complete"
IG_ROOT     = "/kaggle/input/datasets/zlemglsmklkaya/caai-ig-nb4-complete"
SHAP_ROOT   = "/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete"
OUTPUT_DIR  = "/kaggle/working/nb6_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

data = {}

cc_dir = os.path.join(SPLITS_ROOT, "credit_card")
data["credit_card"] = {s: pd.read_parquet(os.path.join(cc_dir, f"{s}.parquet")) for s in ["id", "id2", "ood"]}
with open(os.path.join(cc_dir, "config.json")) as f:
    CC_FEATURES = json.load(f)["model_features"]

for scen in ["scenario_T", "scenario_A"]:
    d = os.path.join(SPLITS_ROOT, "cic_ids2017", scen)
    data[scen] = {s: pd.read_parquet(os.path.join(d, f"{s}.parquet")) for s in ["id", "id2", "ood"]}
with open(os.path.join(SPLITS_ROOT, "cic_ids2017", "config.json")) as f:
    CIC_FEATURES = json.load(f)["model_features"]

print(f"Credit card features: {len(CC_FEATURES)}, CIC features: {len(CIC_FEATURES)}\n")
for cfg, splits in data.items():
    for s, df in splits.items():
        has_label = "Label" in df.columns
        print(f"{cfg} {s}: {df.shape}, Class={df['Class'].value_counts().to_dict()}, has_Label_col={has_label}")

Credit card features: 29, CIC features: 67

credit_card id: (21731, 31), Class={0: 21595, 1: 136}, has_Label_col=False
credit_card id2: (21731, 31), Class={0: 21595, 1: 136}, has_Label_col=False
credit_card ood: (139490, 31), Class={0: 139289, 1: 201}, has_Label_col=False
scenario_T id: (296457, 82), Class={0: 195004, 1: 101453}, has_Label_col=True
scenario_T id2: (296457, 82), Class={0: 195004, 1: 101453}, has_Label_col=True
scenario_T ood: (1017868, 82), Class={0: 795033, 1: 222835}, has_Label_col=True
scenario_A id: (357584, 82), Class={0: 255042, 1: 102542}, has_Label_col=True
scenario_A id2: (357585, 82), Class={0: 255042, 1: 102543}, has_Label_col=True
scenario_A ood: (615433, 82), Class={0: 394777, 1: 220656}, has_Label_col=True


In [3]:
def load_scores(root, model_dir_name, splits=("id", "id2", "ood")):
    return {s: pd.read_parquet(os.path.join(root, model_dir_name, f"scores_{s}.parquet")) for s in splits}

def verify_vae_config(root, model_dir_name):
    with open(os.path.join(root, model_dir_name, "config.json")) as f:
        cfg = json.load(f)
    output_var = cfg.get("architecture", {}).get("output_variance", "")
    assert "global scalar" in output_var, f"{model_dir_name}: NOT corrected formulation (found: '{output_var}')"
    print(f"  ✓ {model_dir_name}: verified corrected formulation")

scores = {}

# Credit Card
scores["credit_card"] = {
    "ae":  load_scores(OLD_CC_ROOT, "ae_credit_card"),
    "vae": load_scores(NB1_ROOT, "vae_credit_card"),
    "if":  load_scores(NB3_ROOT, "if_credit_card"),
}
verify_vae_config(NB1_ROOT, "vae_credit_card")

# scenario_T
scores["scenario_T"] = {
    "ae":  load_scores(AE_T_ROOT, "ae_cic_scenario_T"),
    "vae": load_scores(NB1_ROOT, "vae_cic_scenario_T"),
    "if":  load_scores(NB3_ROOT, "if_cic_scenario_T"),
}
verify_vae_config(NB1_ROOT, "vae_cic_scenario_T")

# scenario_A
scores["scenario_A"] = {
    "ae":  load_scores(NB1_ROOT, "ae_cic_scenario_A"),
    "vae": load_scores(NB1_ROOT, "vae_cic_scenario_A"),
    "if":  load_scores(NB3_ROOT, "if_cic_scenario_A"),
}
verify_vae_config(NB1_ROOT, "vae_cic_scenario_A")

print("\nAll scores loaded. Sanity check — row counts should match data splits:")
for cfg in scores:
    for model in scores[cfg]:
        for split in scores[cfg][model]:
            n_scores = len(scores[cfg][model][split])
            n_data = len(data[cfg][split])
            match = "✓" if n_scores == n_data else "✗ MISMATCH"
            print(f"  {cfg}/{model}/{split}: scores={n_scores}, data={n_data}  {match}")

  ✓ vae_credit_card: verified corrected formulation
  ✓ vae_cic_scenario_T: verified corrected formulation
  ✓ vae_cic_scenario_A: verified corrected formulation

All scores loaded. Sanity check — row counts should match data splits:
  credit_card/ae/id: scores=21731, data=21731  ✓
  credit_card/ae/id2: scores=21731, data=21731  ✓
  credit_card/ae/ood: scores=139490, data=139490  ✓
  credit_card/vae/id: scores=21731, data=21731  ✓
  credit_card/vae/id2: scores=21731, data=21731  ✓
  credit_card/vae/ood: scores=139490, data=139490  ✓
  credit_card/if/id: scores=21731, data=21731  ✓
  credit_card/if/id2: scores=21731, data=21731  ✓
  credit_card/if/ood: scores=139490, data=139490  ✓
  scenario_T/ae/id: scores=296457, data=296457  ✓
  scenario_T/ae/id2: scores=296457, data=296457  ✓
  scenario_T/ae/ood: scores=1017868, data=1017868  ✓
  scenario_T/vae/id: scores=296457, data=296457  ✓
  scenario_T/vae/id2: scores=296457, data=296457  ✓
  scenario_T/vae/ood: scores=1017868, data=1017868  ✓

In [4]:
def load_ig_attributions(root, model_dir_name, splits=("id", "id2", "ood")):
    out = {}
    for s in splits:
        out[s] = {}
        for cls in [0, 1]:
            path = os.path.join(root, model_dir_name, f"attributions_{s}_class{cls}.npy")
            if os.path.exists(path):
                out[s][cls] = np.load(path)
    return out

def load_shap_values(root, model_dir_name, splits=("id", "id2", "ood")):
    out = {}
    for s in splits:
        out[s] = {}
        for cls in [0, 1]:
            path = os.path.join(root, model_dir_name, f"shap_values_{s}_class{cls}.npy")
            if os.path.exists(path):
                out[s][cls] = np.load(path)
    return out

ig = {
    "credit_card": {"ae": load_ig_attributions(IG_ROOT, "ig_ae_credit_card"), "vae": load_ig_attributions(IG_ROOT, "ig_vae_credit_card")},
    "scenario_T":  {"ae": load_ig_attributions(IG_ROOT, "ig_ae_cic_scenario_T"), "vae": load_ig_attributions(IG_ROOT, "ig_vae_cic_scenario_T")},
    "scenario_A":  {"ae": load_ig_attributions(IG_ROOT, "ig_ae_cic_scenario_A"), "vae": load_ig_attributions(IG_ROOT, "ig_vae_cic_scenario_A")},
}

shap_vals = {
    "credit_card": {m: load_shap_values(SHAP_ROOT, f"shap_{m}_credit_card") for m in ["ae", "vae", "if"]},
    "scenario_T":  {m: load_shap_values(SHAP_ROOT, f"shap_{m}_cic_scenario_T") for m in ["ae", "vae", "if"]},
    "scenario_A":  {m: load_shap_values(SHAP_ROOT, f"shap_{m}_cic_scenario_A") for m in ["ae", "vae", "if"]},
}

print("IG shapes (credit_card, ae, id, class 0):", ig["credit_card"]["ae"]["id"][0].shape)
print("SHAP shapes (credit_card, ae, id, class 0):", shap_vals["credit_card"]["ae"]["id"][0].shape)

print("\nFull IG inventory:")
for cfg in ig:
    for model in ig[cfg]:
        for split in ig[cfg][model]:
            for cls in ig[cfg][model][split]:
                print(f"  ig/{cfg}/{model}/{split}/class{cls}: {ig[cfg][model][split][cls].shape}")

print("\nFull SHAP inventory:")
for cfg in shap_vals:
    for model in shap_vals[cfg]:
        for split in shap_vals[cfg][model]:
            for cls in shap_vals[cfg][model][split]:
                print(f"  shap/{cfg}/{model}/{split}/class{cls}: {shap_vals[cfg][model][split][cls].shape}")

IG shapes (credit_card, ae, id, class 0): (200, 29)
SHAP shapes (credit_card, ae, id, class 0): (200, 29)

Full IG inventory:
  ig/credit_card/ae/id/class0: (200, 29)
  ig/credit_card/ae/id/class1: (136, 29)
  ig/credit_card/ae/id2/class0: (200, 29)
  ig/credit_card/ae/id2/class1: (136, 29)
  ig/credit_card/ae/ood/class0: (200, 29)
  ig/credit_card/ae/ood/class1: (200, 29)
  ig/credit_card/vae/id/class0: (200, 29)
  ig/credit_card/vae/id/class1: (136, 29)
  ig/credit_card/vae/id2/class0: (200, 29)
  ig/credit_card/vae/id2/class1: (136, 29)
  ig/credit_card/vae/ood/class0: (200, 29)
  ig/credit_card/vae/ood/class1: (200, 29)
  ig/scenario_T/ae/id/class0: (200, 67)
  ig/scenario_T/ae/id/class1: (200, 67)
  ig/scenario_T/ae/id2/class0: (200, 67)
  ig/scenario_T/ae/id2/class1: (200, 67)
  ig/scenario_T/ae/ood/class0: (200, 67)
  ig/scenario_T/ae/ood/class1: (200, 67)
  ig/scenario_T/vae/id/class0: (200, 67)
  ig/scenario_T/vae/id/class1: (200, 67)
  ig/scenario_T/vae/id2/class0: (200, 67)


In [5]:
# Quick inventory of what's actually in memory from Cells 1-3
import re

print("=== Data splits ===")
for name in sorted(globals().keys()):
    if re.search(r'(split|_id$|_id2$|_ood$|_train$)', name, re.I) and not name.startswith('_'):
        obj = globals()[name]
        if hasattr(obj, 'shape'):
            print(f"  {name}: {type(obj).__name__} shape={obj.shape}")
        elif isinstance(obj, dict):
            print(f"  {name}: dict, keys={list(obj.keys())[:5]}")

print("\n=== Model scores ===")
for name in sorted(globals().keys()):
    if re.search(r'score', name, re.I) and not name.startswith('_'):
        obj = globals()[name]
        if isinstance(obj, dict):
            print(f"  {name}: dict, keys={list(obj.keys())[:8]}")

print("\n=== IG attributions ===")
for name in sorted(globals().keys()):
    if re.search(r'^ig_|attribution', name, re.I) and not name.startswith('_'):
        obj = globals()[name]
        if isinstance(obj, dict):
            print(f"  {name}: dict, keys={list(obj.keys())[:8]}")

print("\n=== SHAP values ===")
for name in sorted(globals().keys()):
    if re.search(r'^shap_|shap_values', name, re.I) and not name.startswith('_'):
        obj = globals()[name]
        if isinstance(obj, dict):
            print(f"  {name}: dict, keys={list(obj.keys())[:8]}")

print("\n=== Loaded model objects (for faithfulness/deletion-AUC re-scoring) ===")
for name in sorted(globals().keys()):
    if re.search(r'^(ae_|vae_|if_)model|^model_', name, re.I) and not name.startswith('_'):
        print(f"  {name}: {type(globals()[name])}")

=== Data splits ===
  splits: dict, keys=['id', 'id2', 'ood']

=== Model scores ===
  scores: dict, keys=['credit_card', 'scenario_T', 'scenario_A']

=== IG attributions ===

=== SHAP values ===
  shap_vals: dict, keys=['credit_card', 'scenario_T', 'scenario_A']

=== Loaded model objects (for faithfulness/deletion-AUC re-scoring) ===


In [6]:
import numpy as np

def describe(obj, depth=0, max_depth=4, prefix=""):
    indent = "  " * depth
    if depth > max_depth:
        print(f"{indent}{prefix}... (max depth)")
        return
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"{indent}{prefix}dict, {len(keys)} keys: {keys}")
        # recurse into first key only, to see the pattern without flooding output
        if keys:
            first_key = keys[0]
            describe(obj[first_key], depth+1, max_depth, prefix=f"['{first_key}'] -> ")
    elif isinstance(obj, np.ndarray):
        print(f"{indent}{prefix}ndarray, shape={obj.shape}, dtype={obj.dtype}")
    elif hasattr(obj, 'shape'):
        print(f"{indent}{prefix}{type(obj).__name__}, shape={obj.shape}")
    else:
        print(f"{indent}{prefix}{type(obj).__name__}: {repr(obj)[:80]}")

print("=" * 60)
print("splits")
print("=" * 60)
describe(splits)

print("\n" + "=" * 60)
print("scores")
print("=" * 60)
describe(scores)

print("\n" + "=" * 60)
print("shap_vals")
print("=" * 60)
describe(shap_vals)

print("\n" + "=" * 60)
print("ALL dict-type globals (to find the IG attributions variable)")
print("=" * 60)
for name, obj in sorted(globals().items()):
    if name.startswith('_'):
        continue
    if isinstance(obj, dict):
        keys_preview = list(obj.keys())[:6]
        print(f"  {name}: dict, {len(obj)} keys, sample={keys_preview}")

print("\n" + "=" * 60)
print("ALL loaded torch/sklearn model objects")
print("=" * 60)
import torch
for name, obj in sorted(globals().items()):
    if name.startswith('_'):
        continue
    if isinstance(obj, torch.nn.Module):
        print(f"  {name}: {type(obj).__name__} (torch model)")
    elif hasattr(obj, 'predict') and hasattr(obj, 'fit') and not isinstance(obj, dict):
        print(f"  {name}: {type(obj).__name__} (sklearn-like model)")

print("\n" + "=" * 60)
print("Feature column names available? (needed for deletion-AUC ranking)")
print("=" * 60)
for name, obj in sorted(globals().items()):
    if name.startswith('_'):
        continue
    if 'feature' in name.lower() or 'col' in name.lower():
        if isinstance(obj, (list, tuple)):
            print(f"  {name}: list[{len(obj)}], first 5 = {list(obj)[:5]}")

splits
dict, 3 keys: ['id', 'id2', 'ood']
  ['id'] -> DataFrame, shape=(357584, 82)

scores
dict, 3 keys: ['credit_card', 'scenario_T', 'scenario_A']
  ['credit_card'] -> dict, 3 keys: ['ae', 'vae', 'if']
    ['ae'] -> dict, 3 keys: ['id', 'id2', 'ood']
      ['id'] -> DataFrame, shape=(21731, 3)

shap_vals
dict, 3 keys: ['credit_card', 'scenario_T', 'scenario_A']
  ['credit_card'] -> dict, 3 keys: ['ae', 'vae', 'if']
    ['ae'] -> dict, 3 keys: ['id', 'id2', 'ood']
      ['id'] -> dict, 2 keys: [0, 1]
        ['0'] -> ndarray, shape=(200, 29), dtype=float64

ALL dict-type globals (to find the IG attributions variable)
  Out: dict, 0 keys, sample=[]
  data: dict, 3 keys, sample=['credit_card', 'scenario_T', 'scenario_A']
  ig: dict, 3 keys, sample=['credit_card', 'scenario_T', 'scenario_A']
  obj: dict, 3 keys, sample=['credit_card', 'scenario_T', 'scenario_A']
  scores: dict, 3 keys, sample=['credit_card', 'scenario_T', 'scenario_A']
  shap_vals: dict, 3 keys, sample=['credit_card', '

In [7]:
import numpy as np

def describe(obj, depth=0, max_depth=5, prefix=""):
    indent = "  " * depth
    if depth > max_depth:
        print(f"{indent}{prefix}... (max depth)")
        return
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"{indent}{prefix}dict, {len(keys)} keys: {keys}")
        if keys:
            first_key = keys[0]
            describe(obj[first_key], depth+1, max_depth, prefix=f"[{first_key!r}] -> ")
    elif isinstance(obj, np.ndarray):
        print(f"{indent}{prefix}ndarray, shape={obj.shape}, dtype={obj.dtype}")
    elif hasattr(obj, 'shape') and hasattr(obj, 'columns'):
        print(f"{indent}{prefix}DataFrame, shape={obj.shape}, columns[:8]={list(obj.columns)[:8]}")
    elif hasattr(obj, 'shape'):
        print(f"{indent}{prefix}{type(obj).__name__}, shape={obj.shape}")
    else:
        print(f"{indent}{prefix}{type(obj).__name__}: {repr(obj)[:100]}")

print("="*60, "\ndata\n", "="*60)
describe(data)

print("\n"+"="*60, "\nig\n", "="*60)
describe(ig)

print("\n"+"="*60, "\nig — does it have an 'if' key anywhere? (should NOT, IF is non-differentiable)\n", "="*60)
for ds in ig:
    print(f"  {ds}: models = {list(ig[ds].keys())}")

print("\n"+"="*60, "\nobj (unclear purpose — inspect fully)\n", "="*60)
describe(obj)

print("\n"+"="*60, "\nscores — full model list per dataset\n", "="*60)
for ds in scores:
    print(f"  {ds}: models = {list(scores[ds].keys())}")

print("\n"+"="*60, "\nsplits — is this stale? compare row counts to data['credit_card']\n", "="*60)
print("splits keys:", list(splits.keys()))
if 'credit_card' in data:
    print("data['credit_card'] keys:", list(data['credit_card'].keys()) if isinstance(data['credit_card'], dict) else type(data['credit_card']))

print("\n"+"="*60, "\nFeature column order check: does shap_vals column count match CC_FEATURES/CIC_FEATURES length?\n", "="*60)
print("CC_FEATURES len:", len(CC_FEATURES))
print("CIC_FEATURES len:", len(CIC_FEATURES))
print("shap_vals['credit_card']['ae']['id'][0].shape:", shap_vals['credit_card']['ae']['id'][0].shape)
print("shap_vals['scenario_T']['ae']['id'][0].shape:", shap_vals['scenario_T']['ae']['id'][0].shape)

print("\n"+"="*60, "\nig attribution shape check (same feature-order question)\n", "="*60)
ds0 = list(ig.keys())[0]
model0 = list(ig[ds0].keys())[0]
split0 = list(ig[ds0][model0].keys())[0]
cls0 = list(ig[ds0][model0][split0].keys())[0]
print(f"ig['{ds0}']['{model0}']['{split0}'][{cls0}].shape =", ig[ds0][model0][split0][cls0].shape)

data
dict, 3 keys: ['credit_card', 'scenario_T', 'scenario_A']
  ['credit_card'] -> dict, 3 keys: ['id', 'id2', 'ood']
    ['id'] -> DataFrame, shape=(21731, 31), columns[:8]=['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7']

ig
dict, 3 keys: ['credit_card', 'scenario_T', 'scenario_A']
  ['credit_card'] -> dict, 2 keys: ['ae', 'vae']
    ['ae'] -> dict, 3 keys: ['id', 'id2', 'ood']
      ['id'] -> dict, 2 keys: [0, 1]
        [0] -> ndarray, shape=(200, 29), dtype=float32

ig — does it have an 'if' key anywhere? (should NOT, IF is non-differentiable)
  credit_card: models = ['ae', 'vae']
  scenario_T: models = ['ae', 'vae']
  scenario_A: models = ['ae', 'vae']

obj (unclear purpose — inspect fully)
function: <function verify_vae_config at 0x7b2850604040>

scores — full model list per dataset
  credit_card: models = ['ae', 'vae', 'if']
  scenario_T: models = ['ae', 'vae', 'if']
  scenario_A: models = ['ae', 'vae', 'if']

splits — is this stale? compare row counts to data['credit_card']

In [8]:
# 1. What columns does `scores` actually have?
print("scores['credit_card']['ae']['id'].columns:", list(scores['credit_card']['ae']['id'].columns))
print(scores['credit_card']['ae']['id'].head(3))
print()
print("scores['scenario_T']['ae']['id'].columns:", list(scores['scenario_T']['ae']['id'].columns))
print(scores['scenario_T']['ae']['id'].head(3))

# 2. Any index/row-tracking variables saved from NB4/NB5?
print("\n--- searching for index/indices tracking ---")
for name, o in sorted(globals().items()):
    if name.startswith('_'):
        continue
    if 'idx' in name.lower() or 'indices' in name.lower() or 'index' in name.lower():
        print(f"  {name}: {type(o)}")

# 3. Do the ig/shap config.json files (if loaded) record which rows were sampled?
for name, o in sorted(globals().items()):
    if name.startswith('_'):
        continue
    if isinstance(o, dict) and name not in ('data','scores','shap_vals','ig','splits'):
        keys_preview = list(o.keys())[:10]
        if any(k in ('n_instances_target','n_instances','seed','random_state','indices','sampled_indices') for k in (keys_preview if all(isinstance(k,str) for k in keys_preview) else [])):
            print(f"  possible config: {name} -> {o}")

# 4. Does data[dataset][split] carry a Label/attack-type column we could match to a Class label?
for ds in ['scenario_T','scenario_A']:
    cols = list(data[ds]['id'].columns)
    print(f"\n{ds} id columns ({len(cols)}):", cols)

# 5. Row count sanity: is 200 a subsample of the full split, and is there a Class column to filter by?
print("\ncredit_card id full shape:", data['credit_card']['id'].shape)
print("credit_card id columns:", list(data['credit_card']['id'].columns))

scores['credit_card']['ae']['id'].columns: ['row_id', 'Class', 'reconstruction_error']
   row_id  Class  reconstruction_error
0       0      0              0.151871
1       1      0              0.326639
2       2      0              0.152467

scores['scenario_T']['ae']['id'].columns: ['row_id', 'Class', 'reconstruction_error']
   row_id  Class  reconstruction_error
0       0      1              4.207468
1       1      0              0.000354
2       2      1              1.082498

--- searching for index/indices tracking ---

scenario_T id columns (82): ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow I

In [9]:
import os

# Find where the ig/shap datasets are mounted
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        if d.startswith('ig_') or d.startswith('shap_') or 'nb4' in d.lower() or 'nb5' in d.lower():
            full = os.path.join(root, d)
            print(full)
    # don't descend too deep / avoid huge output
    if root.count(os.sep) > 4:
        dirs[:] = []

print("\n--- Full file listing for one IG folder ---")
import glob
ig_candidates = glob.glob('/kaggle/input/**/ig_ae_credit_card*', recursive=True)
print("ig_ae_credit_card matches:", ig_candidates)
if ig_candidates:
    for f in sorted(os.listdir(ig_candidates[0])):
        print(" ", f)

print("\n--- Full file listing for one SHAP folder ---")
shap_candidates = glob.glob('/kaggle/input/**/shap_ae_credit_card*', recursive=True)
print("shap_ae_credit_card matches:", shap_candidates)
if shap_candidates:
    for f in sorted(os.listdir(shap_candidates[0])):
        print(" ", f)

print("\n--- Check a config.json for saved indices/seed info ---")
if ig_candidates:
    cfg_path = os.path.join(ig_candidates[0], 'config.json')
    if os.path.exists(cfg_path):
        import json
        with open(cfg_path) as f:
            print(json.load(f))
    else:
        print("no config.json at", cfg_path)

/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete
/kaggle/input/datasets/zlemglsmklkaya/caai-ig-nb4-complete
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_if_cic_scenario_T
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_if_cic_scenario_A
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_vae_cic_scenario_A
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_vae_cic_scenario_T
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_ae_cic_scenario_T
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_if_credit_card
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_ae_credit_card
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_ae_cic_scenario_A
/kaggle/input/datasets/zlemglsmklkaya/caai-shap-nb5-complete/shap_vae_credit_card
/kaggle/input/datasets/zlemglsmklkaya/caai-ig-nb4-complete/ig_vae_cic_scenario_T
/kaggle/input/datasets/zlemglsmklkaya/caai-ig-nb4

In [10]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

def paired_spearman_stability(arr_a, arr_b):
    """
    Pairs row i of arr_a with row i of arr_b (index-position pairing).
    Grounded in manuscript Sec 3.5: 'attribution vectors are obtained from
    comparable ID/OOD samples... Spearman correlation is then computed
    between corresponding attribution rankings.' Table 1 reports mean±std
    across a population of such pair-level correlations, not a single
    aggregate-ranking correlation -- confirming per-pair computation.
    Returns one Spearman r per pair (correlation across the feature axis).
    """
    n = min(len(arr_a), len(arr_b))
    corrs = np.empty(n)
    for i in range(n):
        r, _ = spearmanr(arr_a[i], arr_b[i])
        corrs[i] = r
    return corrs

results = []
datasets = ['credit_card', 'scenario_T', 'scenario_A']
methods = {
    'ig':   {'dict': ig,        'models': ['ae', 'vae']},
    'shap': {'dict': shap_vals, 'models': ['ae', 'vae', 'if']},
}
comparisons = [('id', 'ood', 'ID_vs_OOD'), ('id', 'id2', 'ID_vs_ID2')]

for ds in datasets:
    for method_name, meta in methods.items():
        method_dict = meta['dict']
        for model in meta['models']:
            if model not in method_dict.get(ds, {}):
                continue
            for cls in [0, 1]:
                for split_a, split_b, comp_name in comparisons:
                    try:
                        arr_a = method_dict[ds][model][split_a][cls]
                        arr_b = method_dict[ds][model][split_b][cls]
                    except KeyError:
                        continue
                    corrs = paired_spearman_stability(arr_a, arr_b)
                    results.append({
                        'dataset': ds, 'method': method_name, 'model': model,
                        'class': cls, 'comparison': comp_name,
                        'n_pairs': len(corrs),
                        'mean_spearman': np.mean(corrs),
                        'std_spearman': np.std(corrs),
                        'median_spearman': np.median(corrs),
                    })

stability_df = pd.DataFrame(results)
pd.set_option('display.width', 160)
pd.set_option('display.max_rows', 200)
print(stability_df.to_string(index=False))

print("\n" + "="*70)
print("SANITY CHECK vs published manuscript Table 1 (IG, AE, credit_card, ID_vs_OOD)")
print("Table 1 says: Benign 0.060 ± 0.225   |   Anomaly 0.264 ± 0.255")
print("="*70)
check = stability_df[(stability_df.dataset=='credit_card') & (stability_df.method=='ig') &
                      (stability_df.model=='ae') & (stability_df.comparison=='ID_vs_OOD')]
print(check[['class','n_pairs','mean_spearman','std_spearman']].to_string(index=False))

stability_df.to_csv('/kaggle/working/nb6_stability_results.csv', index=False)
print("\nSaved: /kaggle/working/nb6_stability_results.csv")

    dataset method model  class comparison  n_pairs  mean_spearman  std_spearman  median_spearman
credit_card     ig    ae      0  ID_vs_OOD      200       0.039621      0.237010         0.033744
credit_card     ig    ae      0  ID_vs_ID2      200       0.081244      0.238422         0.080788
credit_card     ig    ae      1  ID_vs_OOD      136       0.364938      0.301311         0.422167
credit_card     ig    ae      1  ID_vs_ID2      136       0.412605      0.325240         0.442611
credit_card     ig   vae      0  ID_vs_OOD      200       0.080357      0.198823         0.088424
credit_card     ig   vae      0  ID_vs_ID2      200       0.129163      0.214014         0.148030
credit_card     ig   vae      1  ID_vs_OOD      136       0.436482      0.276700         0.485961
credit_card     ig   vae      1  ID_vs_ID2      136       0.495001      0.300722         0.524631
credit_card   shap    ae      0  ID_vs_OOD      200       0.004736      0.217426         0.002055
credit_card   shap  

In [11]:
import os, json

for name, path_hint in [
    ('ae_credit_card', 'caai-revision-splits-new'),
    ('vae_credit_card', 'caai-models-nb1-complete'),
    ('ae_cic_scenario_T', 'caai-models-cic-scenario-t'),
    ('vae_cic_scenario_T', 'caai-models-nb1-complete'),
    ('ae_cic_scenario_A', 'caai-models-nb1-complete'),
    ('vae_cic_scenario_A', 'caai-models-nb1-complete'),
    ('if_credit_card', 'caai-models-nb3-complete'),
    ('if_cic_scenario_T', 'caai-models-nb3-complete'),
    ('if_cic_scenario_A', 'caai-models-nb3-complete'),
]:
    import glob
    matches = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    print(f"\n--- {name} ---")
    if not matches:
        print("  NOT FOUND")
        continue
    folder = matches[0]
    print(" folder:", folder)
    for f in sorted(os.listdir(folder)):
        fpath = os.path.join(folder, f)
        print("  ", f, f"({os.path.getsize(fpath)} bytes)")
    cfg_path = os.path.join(folder, 'config.json')
    if os.path.exists(cfg_path):
        with open(cfg_path) as fh:
            cfg = json.load(fh)
        print("  config.json contents:", json.dumps(cfg, indent=2)[:2000])


--- ae_credit_card ---
 folder: /kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits-new/ae_credit_card
   config.json (699 bytes)
   model.pt (12635 bytes)
   scores_id.parquet (257699 bytes)
   scores_id2.parquet (257759 bytes)
   scores_ood.parquet (1662576 bytes)
  config.json contents: {
  "model": "Autoencoder",
  "dataset": "credit_card",
  "architecture": {
    "hidden_dims": [
      20,
      14
    ],
    "latent_dim": 7,
    "architecture_rationale": "Shallow funnel (29->20->14->7->14->20->29); width chosen relative to the 29 input features, not tuned"
  },
  "training": {
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "batch_size": 256,
    "max_epochs": 300,
    "patience": 10,
    "min_delta": 1e-05,
    "early_stopping_criterion": "validation MSE, no improvement > min_delta for `patience` consecutive epochs",
    "seed": 42,
    "loss": "MSE"
  },
  "stopped_epoch": 170,
  "best_val_loss": 0.29996949434280396,
  "fit_size": 90697,
  "val_size": 10077
}



In [12]:
import torch

for name, path in [
    ('ae_credit_card', '/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits-new/ae_credit_card/model.pt'),
    ('vae_credit_card', '/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb1-complete/vae_credit_card/model.pt'),
]:
    print(f"\n=== {name} ===")
    obj = torch.load(path, map_location='cpu')
    print("Top-level type:", type(obj))
    if isinstance(obj, dict):
        print("Top-level keys:", list(obj.keys()))
        # if it's wrapped (e.g. {'state_dict':..., 'config':...}), drill in
        sd = obj.get('state_dict', obj) if 'state_dict' in obj else obj
        # Might also be directly a state_dict of tensors
        for k, v in sd.items():
            if hasattr(v, 'shape'):
                print(f"  {k}: {tuple(v.shape)}")
            else:
                print(f"  {k}: {type(v)} (not a tensor -- nested dict?)")
                if isinstance(v, dict):
                    for k2, v2 in v.items():
                        shape = tuple(v2.shape) if hasattr(v2, 'shape') else type(v2)
                        print(f"      {k2}: {shape}")


=== ae_credit_card ===
Top-level type: <class 'collections.OrderedDict'>
Top-level keys: ['encoder.0.weight', 'encoder.0.bias', 'encoder.2.weight', 'encoder.2.bias', 'encoder.4.weight', 'encoder.4.bias', 'decoder.0.weight', 'decoder.0.bias', 'decoder.2.weight', 'decoder.2.bias', 'decoder.4.weight', 'decoder.4.bias']
  encoder.0.weight: (20, 29)
  encoder.0.bias: (20,)
  encoder.2.weight: (14, 20)
  encoder.2.bias: (14,)
  encoder.4.weight: (7, 14)
  encoder.4.bias: (7,)
  decoder.0.weight: (14, 7)
  decoder.0.bias: (14,)
  decoder.2.weight: (20, 14)
  decoder.2.bias: (20,)
  decoder.4.weight: (29, 20)
  decoder.4.bias: (29,)

=== vae_credit_card ===
Top-level type: <class 'collections.OrderedDict'>
Top-level keys: ['global_logvar', 'encoder_body.0.weight', 'encoder_body.0.bias', 'encoder_body.2.weight', 'encoder_body.2.bias', 'fc_mu.weight', 'fc_mu.bias', 'fc_logvar.weight', 'fc_logvar.bias', 'decoder_body.0.weight', 'decoder_body.0.bias', 'decoder_body.2.weight', 'decoder_body.2.bias

In [14]:
feat_cols = CC_FEATURES  # exact 29 features used in training, excludes Time/Class
df_id = data['credit_card']['id'].reset_index(drop=True)
n_check = 500
X_check = torch.tensor(df_id[feat_cols].values[:n_check], dtype=torch.float32)

print("Using feat_cols:", feat_cols)
print("X_check shape:", X_check.shape)

stored_ae = scores['credit_card']['ae']['id'].iloc[:n_check]['reconstruction_error'].values
vae_score_col = 'reconstruction_error' if 'reconstruction_error' in scores['credit_card']['vae']['id'].columns else scores['credit_card']['vae']['id'].columns[-1]
stored_vae = scores['credit_card']['vae']['id'].iloc[:n_check][vae_score_col].values
stored_if = scores['credit_card']['if']['id'].iloc[:n_check]

my_ae_mean = ae_mse_mean(ae_cc, X_check)
my_ae_sum = ae_mse_sum(ae_cc, X_check)
my_vae = vae_nll(vae_cc, X_check)
my_if = -if_cc.score_samples(df_id[feat_cols].values[:n_check])

from scipy.stats import pearsonr
print("\n--- AE verification ---")
print("corr(mean-MSE, stored):", pearsonr(my_ae_mean, stored_ae)[0])
print("corr(sum-MSE, stored):", pearsonr(my_ae_sum, stored_ae)[0])
print("ratio stored/mean (median):", np.median(stored_ae / my_ae_mean))
print("ratio stored/sum (median):", np.median(stored_ae / my_ae_sum))
print("sample stored vs mean-MSE vs sum-MSE:", list(zip(stored_ae[:5], my_ae_mean[:5], my_ae_sum[:5])))

print("\n--- VAE verification ---")
print("stored vae score column used:", vae_score_col)
print("corr(my_vae_nll, stored):", pearsonr(my_vae, stored_vae)[0])
print("ratio stored/mine (median):", np.median(stored_vae / my_vae))
print("sample stored vs mine:", list(zip(stored_vae[:5], my_vae[:5])))

print("\n--- IF verification ---")
print("stored IF columns:", list(stored_if.columns))
stored_if_score = stored_if[stored_if.columns[-1]].values
print("corr(my_if, stored):", pearsonr(my_if, stored_if_score)[0])
print("max abs diff:", np.max(np.abs(my_if - stored_if_score)))

Using feat_cols: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']
X_check shape: torch.Size([500, 29])

--- AE verification ---
corr(mean-MSE, stored): 1.0
corr(sum-MSE, stored): 1.0
ratio stored/mean (median): 1.0
ratio stored/sum (median): 0.03448276
sample stored vs mean-MSE vs sum-MSE: [(np.float32(0.1518714), np.float32(0.15187141), np.float32(4.404271)), (np.float32(0.32663938), np.float32(0.32663938), np.float32(9.472542)), (np.float32(0.1524674), np.float32(0.1524674), np.float32(4.4215546)), (np.float32(0.35403317), np.float32(0.35403317), np.float32(10.266962)), (np.float32(0.21874984), np.float32(0.21874984), np.float32(6.343745))]

--- VAE verification ---
stored vae score column used: vae_nll
corr(my_vae_nll, stored): 0.99943894
ratio stored/mine (median): 1.0002403
sample stored vs mine: [(np.float32(60.340717), np.float32(60

In [15]:
def load_ae(input_dim, hidden_dims, latent_dim, path):
    m = Autoencoder(input_dim, hidden_dims, latent_dim)
    m.load_state_dict(torch.load(path, map_location='cpu'))
    m.eval()
    return m

def load_vae(input_dim, hidden_dims, latent_dim, path):
    m = VAE(input_dim, hidden_dims, latent_dim)
    m.load_state_dict(torch.load(path, map_location='cpu'))
    m.eval()
    return m

base = '/kaggle/input/datasets/zlemglsmklkaya'

models = {
    'credit_card': {
        'ae': ae_cc,   # already loaded/verified
        'vae': vae_cc, # already loaded/verified
        'if': if_cc,   # already loaded/verified
    },
    'scenario_T': {
        'ae': load_ae(67, [48, 32], 16, f'{base}/caai-models-cic-scenario-t/ae_cic_scenario_T/model.pt'),
        'vae': load_vae(67, [48, 32], 16, f'{base}/caai-models-nb1-complete/vae_cic_scenario_T/model.pt'),
        'if': joblib.load(f'{base}/caai-models-nb3-complete/if_cic_scenario_T/model.joblib'),
    },
    'scenario_A': {
        'ae': load_ae(67, [48, 32], 16, f'{base}/caai-models-nb1-complete/ae_cic_scenario_A/model.pt'),
        'vae': load_vae(67, [48, 32], 16, f'{base}/caai-models-nb1-complete/vae_cic_scenario_A/model.pt'),
        'if': joblib.load(f'{base}/caai-models-nb3-complete/if_cic_scenario_A/model.joblib'),
    },
}
print("All 9 models loaded.")

# ---- Verify scenario_T and scenario_A (AE, VAE, IF) against stored scores ----
from scipy.stats import pearsonr, spearmanr

for ds in ['scenario_T', 'scenario_A']:
    print(f"\n{'='*50}\n{ds}\n{'='*50}")
    df_id = data[ds]['id'].reset_index(drop=True)
    n_check = 300
    X_check = torch.tensor(df_id[CIC_FEATURES].values[:n_check], dtype=torch.float32)

    stored_ae = scores[ds]['ae']['id'].iloc[:n_check]['reconstruction_error'].values
    vae_col = 'reconstruction_error' if 'reconstruction_error' in scores[ds]['vae']['id'].columns else scores[ds]['vae']['id'].columns[-1]
    stored_vae = scores[ds]['vae']['id'].iloc[:n_check][vae_col].values
    stored_if_df = scores[ds]['if']['id'].iloc[:n_check]
    stored_if_score = stored_if_df[stored_if_df.columns[-1]].values

    my_ae = ae_mse_mean(models[ds]['ae'], X_check)
    my_vae = vae_nll(models[ds]['vae'], X_check)
    my_if = -models[ds]['if'].score_samples(df_id[CIC_FEATURES].values[:n_check])

    print("AE  : pearson =", pearsonr(my_ae, stored_ae)[0], " spearman =", spearmanr(my_ae, stored_ae)[0])
    print("VAE : pearson =", pearsonr(my_vae, stored_vae)[0], " spearman =", spearmanr(my_vae, stored_vae)[0])
    print("IF  : pearson =", pearsonr(my_if, stored_if_score)[0], " max_abs_diff =", np.max(np.abs(my_if - stored_if_score)))

All 9 models loaded.

scenario_T
AE  : pearson = 1.0  spearman = 1.0
VAE : pearson = 0.9999449  spearman = 0.9696854409493438
IF  : pearson = 0.9999999999999998  max_abs_diff = 1.1102230246251565e-16

scenario_A
AE  : pearson = 0.9999998  spearman = 1.0
VAE : pearson = 0.9998522  spearman = 0.9748530539228212
IF  : pearson = 0.9999999999999997  max_abs_diff = 1.1102230246251565e-16


In [16]:
import torch

FAITH_N = 50          # instances per (dataset, split, class) group — deletion-AUC needs less than stability's n=200
FAITH_SEED = 42

faith_samples = {}

for ds in ['credit_card', 'scenario_T', 'scenario_A']:
    faith_samples[ds] = {}
    feat_cols = CC_FEATURES if ds == 'credit_card' else CIC_FEATURES
    for split in ['id', 'id2', 'ood']:
        faith_samples[ds][split] = {}
        df = data[ds][split].reset_index(drop=True)
        df['row_id'] = df.index  # explicit, so we never lose this again
        for cls in [0, 1]:
            sub = df[df['Class'] == cls]
            n = min(FAITH_N, len(sub))
            sampled = sub.sample(n=n, random_state=FAITH_SEED + cls)  # fixed, reproducible
            X = torch.tensor(sampled[feat_cols].values, dtype=torch.float32)
            faith_samples[ds][split][cls] = {
                'row_id': sampled['row_id'].values,
                'X': X,
                'feat_cols': feat_cols,
            }

# Summary
print(f"{'dataset':<14}{'split':<8}{'class':<7}{'n':<5}")
for ds in faith_samples:
    for split in faith_samples[ds]:
        for cls in faith_samples[ds][split]:
            n = len(faith_samples[ds][split][cls]['row_id'])
            print(f"{ds:<14}{split:<8}{cls:<7}{n:<5}")

# Sanity: row_ids are genuinely traceable back to data
ds, split, cls = 'credit_card', 'id', 1
rid = faith_samples[ds][split][cls]['row_id'][0]
x_from_sample = faith_samples[ds][split][cls]['X'][0].numpy()
x_from_data = data[ds][split].reset_index(drop=True).iloc[rid][CC_FEATURES].values
print("\nTraceability check (should be True):", np.allclose(x_from_sample, x_from_data))

dataset       split   class  n    
credit_card   id      0      50   
credit_card   id      1      50   
credit_card   id2     0      50   
credit_card   id2     1      50   
credit_card   ood     0      50   
credit_card   ood     1      50   
scenario_T    id      0      50   
scenario_T    id      1      50   
scenario_T    id2     0      50   
scenario_T    id2     1      50   
scenario_T    ood     0      50   
scenario_T    ood     1      50   
scenario_A    id      0      50   
scenario_A    id      1      50   
scenario_A    id2     0      50   
scenario_A    id2     1      50   
scenario_A    ood     0      50   
scenario_A    ood     1      50   

Traceability check (should be True): True


In [18]:
!pip install captum --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 21.9 MB/s eta 0:00:00


In [19]:
from captum.attr import IntegratedGradients
import numpy as np
import torch

IG_N_STEPS = 300
IG_EXCLUSION_THRESHOLD = 0.2
IG_ABS_DELTA_FLOOR = 0.001

def ae_score_fn(model):
    def f(x):
        recon = model(x)
        return ((x - recon) ** 2).mean(dim=1)
    return f

def vae_score_fn_deterministic(model):
    # z = mu (no sampling) -- deterministic, needed for IG's convergence-delta guarantee.
    # Verified corr ~0.999 vs the MC-10-averaged score used for the main detection tables.
    def f(x):
        mu, logvar_z = model.encode(x)
        out_logvar = torch.clamp(model.global_logvar, -10, 10)
        sigma2 = torch.exp(out_logvar)
        D = x.shape[1]
        recon = model.decode(mu)
        sq_err = ((x - recon) ** 2).sum(dim=1)
        nll = 0.5 * D * torch.log(2 * torch.pi * sigma2) + 0.5 * sq_err / sigma2
        return nll
    return f

faith_ig = {}

for ds in ['credit_card', 'scenario_T', 'scenario_A']:
    faith_ig[ds] = {}
    for model_name in ['ae', 'vae']:
        faith_ig[ds][model_name] = {}
        model = models[ds][model_name]
        score_fn = ae_score_fn(model) if model_name == 'ae' else vae_score_fn_deterministic(model)
        ig = IntegratedGradients(score_fn)
        for split in ['id', 'id2', 'ood']:
            faith_ig[ds][model_name][split] = {}
            for cls in [0, 1]:
                X = faith_samples[ds][split][cls]['X']
                baseline = torch.zeros_like(X)
                attrs, delta = ig.attribute(X, baselines=baseline, n_steps=IG_N_STEPS,
                                             return_convergence_delta=True)
                attrs = attrs.detach().numpy()
                delta = delta.detach().numpy()

                with torch.no_grad():
                    f_x = score_fn(X).numpy()
                    f_baseline = score_fn(baseline).numpy()
                rel_delta = np.abs(delta) / (np.abs(f_x - f_baseline) + 1e-8)
                abs_delta = np.abs(delta)
                excluded = (rel_delta > IG_EXCLUSION_THRESHOLD) & (abs_delta > IG_ABS_DELTA_FLOOR)

                faith_ig[ds][model_name][split][cls] = {
                    'attrs': attrs[~excluded],
                    'row_id': faith_samples[ds][split][cls]['row_id'][~excluded],
                    'X': X[~excluded].numpy(),
                    'n_total': len(X),
                    'n_excluded': int(excluded.sum()),
                }

print(f"{'dataset':<14}{'model':<6}{'split':<8}{'class':<7}{'n_total':<9}{'n_excluded':<11}{'n_final':<8}")
for ds in faith_ig:
    for m in faith_ig[ds]:
        for split in faith_ig[ds][m]:
            for cls in faith_ig[ds][m][split]:
                d = faith_ig[ds][m][split][cls]
                print(f"{ds:<14}{m:<6}{split:<8}{cls:<7}{d['n_total']:<9}{d['n_excluded']:<11}{d['n_total']-d['n_excluded']:<8}")

dataset       model split   class  n_total  n_excluded n_final 
credit_card   ae    id      0      50       7          43      
credit_card   ae    id      1      50       0          50      
credit_card   ae    id2     0      50       8          42      
credit_card   ae    id2     1      50       3          47      
credit_card   ae    ood     0      50       0          50      
credit_card   ae    ood     1      50       0          50      
credit_card   vae   id      0      50       1          49      
credit_card   vae   id      1      50       0          50      
credit_card   vae   id2     0      50       0          50      
credit_card   vae   id2     1      50       0          50      
credit_card   vae   ood     0      50       0          50      
credit_card   vae   ood     1      50       0          50      
scenario_T    ae    id      0      50       0          50      
scenario_T    ae    id      1      50       0          50      
scenario_T    ae    id2     0      50   

In [20]:
import shap
import time

SHAP_BACKGROUND_SIZE = 50
SHAP_NSAMPLES = 'auto'

def make_predict_fn(model, model_name):
    if model_name == 'ae':
        def f(X_np):
            X_t = torch.tensor(X_np, dtype=torch.float32)
            with torch.no_grad():
                recon = model(X_t)
                return ((X_t - recon) ** 2).mean(dim=1).numpy()
        return f
    elif model_name == 'vae':
        def f(X_np):
            X_t = torch.tensor(X_np, dtype=torch.float32)
            with torch.no_grad():
                mu, logvar_z = model.encode(X_t)
                out_logvar = torch.clamp(model.global_logvar, -10, 10)
                sigma2 = torch.exp(out_logvar)
                D = X_t.shape[1]
                recon = model.decode(mu)
                sq_err = ((X_t - recon) ** 2).sum(dim=1)
                nll = 0.5 * D * torch.log(2 * torch.pi * sigma2) + 0.5 * sq_err / sigma2
                return nll.numpy()
        return f
    elif model_name == 'if':
        def f(X_np):
            return -model.score_samples(X_np)
        return f

faith_shap = {}

ds = 'credit_card'
faith_shap[ds] = {}
feat_cols = CC_FEATURES
bg_df = data[ds]['id'].reset_index(drop=True)
bg_df = bg_df[bg_df['Class'] == 0]
background = bg_df[feat_cols].sample(n=SHAP_BACKGROUND_SIZE, random_state=42).values

t_start_all = time.time()
for model_name in ['ae', 'vae', 'if']:
    faith_shap[ds][model_name] = {}
    model = models[ds][model_name]
    predict_fn = make_predict_fn(model, model_name)
    explainer = shap.KernelExplainer(predict_fn, background)

    t0 = time.time()
    for split in ['id', 'id2', 'ood']:
        faith_shap[ds][model_name][split] = {}
        for cls in [0, 1]:
            X = faith_samples[ds][split][cls]['X'].numpy()
            sv = explainer.shap_values(X, nsamples=SHAP_NSAMPLES, silent=True)
            faith_shap[ds][model_name][split][cls] = {
                'shap_values': sv,
                'row_id': faith_samples[ds][split][cls]['row_id'],
            }
    print(f"{ds} / {model_name}: {time.time()-t0:.1f}s for all 6 (split,class) groups (300 instances)")

print(f"\nTotal credit_card SHAP time: {time.time()-t_start_all:.1f}s")

credit_card / ae: 22.8s for all 6 (split,class) groups (300 instances)
credit_card / vae: 23.1s for all 6 (split,class) groups (300 instances)
credit_card / if: 112.2s for all 6 (split,class) groups (300 instances)

Total credit_card SHAP time: 158.1s


In [21]:
ds = 'scenario_T'
faith_shap[ds] = {}
feat_cols = CIC_FEATURES
bg_df = data[ds]['id'].reset_index(drop=True)
bg_df = bg_df[bg_df['Class'] == 0]
background = bg_df[feat_cols].sample(n=SHAP_BACKGROUND_SIZE, random_state=42).values

t_start_all = time.time()
for model_name in ['ae', 'vae', 'if']:
    faith_shap[ds][model_name] = {}
    model = models[ds][model_name]
    predict_fn = make_predict_fn(model, model_name)
    explainer = shap.KernelExplainer(predict_fn, background)

    t0 = time.time()
    for split in ['id', 'id2', 'ood']:
        faith_shap[ds][model_name][split] = {}
        for cls in [0, 1]:
            X = faith_samples[ds][split][cls]['X'].numpy()
            sv = explainer.shap_values(X, nsamples=SHAP_NSAMPLES, silent=True)
            faith_shap[ds][model_name][split][cls] = {
                'shap_values': sv,
                'row_id': faith_samples[ds][split][cls]['row_id'],
            }
    print(f"{ds} / {model_name}: {time.time()-t0:.1f}s for all 6 (split,class) groups (300 instances)")

print(f"\nTotal {ds} SHAP time: {time.time()-t_start_all:.1f}s")

scenario_T / ae: 40.6s for all 6 (split,class) groups (300 instances)
scenario_T / vae: 40.6s for all 6 (split,class) groups (300 instances)
scenario_T / if: 144.5s for all 6 (split,class) groups (300 instances)

Total scenario_T SHAP time: 225.7s


In [22]:
ds = 'scenario_A'
faith_shap[ds] = {}
feat_cols = CIC_FEATURES
bg_df = data[ds]['id'].reset_index(drop=True)
bg_df = bg_df[bg_df['Class'] == 0]
background = bg_df[feat_cols].sample(n=SHAP_BACKGROUND_SIZE, random_state=42).values

t_start_all = time.time()
for model_name in ['ae', 'vae', 'if']:
    faith_shap[ds][model_name] = {}
    model = models[ds][model_name]
    predict_fn = make_predict_fn(model, model_name)
    explainer = shap.KernelExplainer(predict_fn, background)

    t0 = time.time()
    for split in ['id', 'id2', 'ood']:
        faith_shap[ds][model_name][split] = {}
        for cls in [0, 1]:
            X = faith_samples[ds][split][cls]['X'].numpy()
            sv = explainer.shap_values(X, nsamples=SHAP_NSAMPLES, silent=True)
            faith_shap[ds][model_name][split][cls] = {
                'shap_values': sv,
                'row_id': faith_samples[ds][split][cls]['row_id'],
            }
    print(f"{ds} / {model_name}: {time.time()-t0:.1f}s for all 6 (split,class) groups (300 instances)")

print(f"\nTotal {ds} SHAP time: {time.time()-t_start_all:.1f}s")

# All KernelSHAP done -- quick inventory check
print("\n--- Full faith_shap inventory ---")
for d in faith_shap:
    for m in faith_shap[d]:
        n_groups = sum(len(faith_shap[d][m][s]) for s in faith_shap[d][m])
        print(f"  {d} / {m}: {len(faith_shap[d][m])} splits, {n_groups} (split,class) groups")

scenario_A / ae: 38.4s for all 6 (split,class) groups (300 instances)
scenario_A / vae: 37.2s for all 6 (split,class) groups (300 instances)
scenario_A / if: 140.1s for all 6 (split,class) groups (300 instances)

Total scenario_A SHAP time: 215.7s

--- Full faith_shap inventory ---
  credit_card / ae: 3 splits, 6 (split,class) groups
  credit_card / vae: 3 splits, 6 (split,class) groups
  credit_card / if: 3 splits, 6 (split,class) groups
  scenario_T / ae: 3 splits, 6 (split,class) groups
  scenario_T / vae: 3 splits, 6 (split,class) groups
  scenario_T / if: 3 splits, 6 (split,class) groups
  scenario_A / ae: 3 splits, 6 (split,class) groups
  scenario_A / vae: 3 splits, 6 (split,class) groups
  scenario_A / if: 3 splits, 6 (split,class) groups


In [23]:
import numpy as np
import pandas as pd

def ae_score_numpy(model):
    def f(X_np):
        X_t = torch.tensor(X_np, dtype=torch.float32)
        with torch.no_grad():
            recon = model(X_t)
            return ((X_t - recon) ** 2).mean(dim=1).numpy()
    return f

def vae_score_numpy(model):
    def f(X_np):
        X_t = torch.tensor(X_np, dtype=torch.float32)
        with torch.no_grad():
            mu, logvar_z = model.encode(X_t)
            out_logvar = torch.clamp(model.global_logvar, -10, 10)
            sigma2 = torch.exp(out_logvar)
            D = X_t.shape[1]
            recon = model.decode(mu)
            sq_err = ((X_t - recon) ** 2).sum(dim=1)
            nll = 0.5 * D * torch.log(2 * torch.pi * sigma2) + 0.5 * sq_err / sigma2
            return nll.numpy()
    return f

def if_score_numpy(model):
    def f(X_np):
        return -model.score_samples(X_np)
    return f

def compute_deletion_auc(X, attrs, score_fn_numpy, fractions=None):
    """X, attrs: (n,d) numpy arrays, same instance order. Returns per-instance deletion-AUC
    of |score(masked) - score(original)| vs fraction of top-|attribution| features removed
    (set to 0, i.e. the standardized feature mean). Higher AUC = more faithful (unambiguous
    direction, resolving Reviewer 2's Fig.4 axis-clarity request by construction)."""
    if fractions is None:
        fractions = np.linspace(0, 1, 11)
    n, d = X.shape
    orig_scores = score_fn_numpy(X)
    order = np.argsort(-np.abs(attrs), axis=1)
    delta_matrix = np.zeros((n, len(fractions)))
    for j, f in enumerate(fractions):
        k = int(np.ceil(f * d))
        X_masked = X.copy()
        if k > 0:
            for i in range(n):
                X_masked[i, order[i, :k]] = 0.0
        s_masked = score_fn_numpy(X_masked)
        delta_matrix[:, j] = np.abs(s_masked - orig_scores)
    return np.trapz(delta_matrix, fractions, axis=1)

score_fn_builders = {'ae': ae_score_numpy, 'vae': vae_score_numpy, 'if': if_score_numpy}

faith_results = []

# --- IG-based faithfulness (ae, vae) ---
for ds in ['credit_card', 'scenario_T', 'scenario_A']:
    for model_name in ['ae', 'vae']:
        score_fn = score_fn_builders[model_name](models[ds][model_name])
        for split in ['id', 'id2', 'ood']:
            for cls in [0, 1]:
                d = faith_ig[ds][model_name][split][cls]
                if len(d['row_id']) == 0:
                    continue
                aucs = compute_deletion_auc(d['X'], d['attrs'], score_fn)
                faith_results.append({
                    'dataset': ds, 'method': 'ig', 'model': model_name,
                    'split': split, 'class': cls, 'n': len(aucs),
                    'mean_auc': np.mean(aucs), 'std_auc': np.std(aucs),
                    'median_auc': np.median(aucs),
                })

# --- SHAP-based faithfulness (ae, vae, if) ---
for ds in ['credit_card', 'scenario_T', 'scenario_A']:
    feat_cols = CC_FEATURES if ds == 'credit_card' else CIC_FEATURES
    for model_name in ['ae', 'vae', 'if']:
        score_fn = score_fn_builders[model_name](models[ds][model_name])
        for split in ['id', 'id2', 'ood']:
            for cls in [0, 1]:
                d = faith_shap[ds][model_name][split][cls]
                X = faith_samples[ds][split][cls]['X'].numpy()
                sv = np.asarray(d['shap_values'])
                aucs = compute_deletion_auc(X, sv, score_fn)
                faith_results.append({
                    'dataset': ds, 'method': 'shap', 'model': model_name,
                    'split': split, 'class': cls, 'n': len(aucs),
                    'mean_auc': np.mean(aucs), 'std_auc': np.std(aucs),
                    'median_auc': np.median(aucs),
                })

faithfulness_df = pd.DataFrame(faith_results)
pd.set_option('display.width', 160)
pd.set_option('display.max_rows', 250)
print(faithfulness_df.to_string(index=False))

faithfulness_df.to_csv('/kaggle/working/nb6_faithfulness_results.csv', index=False)
print("\nSaved: /kaggle/working/nb6_faithfulness_results.csv")

# Quick eyeball: id vs id2 vs ood, mean_auc pivoted
print("\n--- Quick pivot: mean deletion-AUC by split ---")
pivot = faithfulness_df.pivot_table(index=['dataset','method','model','class'], columns='split', values='mean_auc')
pivot = pivot[['id','id2','ood']] if all(c in pivot.columns for c in ['id','id2','ood']) else pivot
print(pivot.to_string())

/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or o

    dataset method model split  class  n    mean_auc     std_auc  median_auc
credit_card     ig    ae    id      0 43    0.147849    0.107246    0.110633
credit_card     ig    ae    id      1 50   19.885923   22.367462   11.917664
credit_card     ig    ae   id2      0 42    0.151742    0.106927    0.118326
credit_card     ig    ae   id2      1 47   18.447802   21.661043    5.363838
credit_card     ig    ae   ood      0 50    0.356326    0.296899    0.286753
credit_card     ig    ae   ood      1 50   16.124127   21.506717    8.485901
credit_card     ig   vae    id      0 49    7.295844    6.790498    5.146672
credit_card     ig   vae    id      1 50  448.537050  464.519485  291.146200
credit_card     ig   vae   id2      0 50    8.388129    4.833666    8.105222
credit_card     ig   vae   id2      1 50  392.947062  454.722750  194.835828
credit_card     ig   vae   ood      0 50   14.673726    6.971008   13.309283
credit_card     ig   vae   ood      1 50  444.387904  633.558504  203.384016

/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)


In [24]:
import zipfile, os

# Re-save both result tables (already done, but confirm)
stability_df.to_csv('/kaggle/working/nb6_stability_results.csv', index=False)
faithfulness_df.to_csv('/kaggle/working/nb6_faithfulness_results.csv', index=False)

# Also save a config/metadata file documenting the methodology decisions made in this notebook
import json
nb6_metadata = {
    'stability_method': 'Paired index-position Spearman correlation between ID/OOD (and ID/ID2) '
                         'attribution vectors, per manuscript Sec 3.5. n_pairs = min(len(split_a), len(split_b)), '
                         'capped at the stored sample size (200, or 136 for credit_card anomaly class). '
                         'Verified against published Table 1 (IG/AE/credit_card/ID-vs-OOD): '
                         'computed 0.040+/-0.237 (benign) & 0.365+/-0.301 (anomaly) vs published '
                         '0.060+/-0.225 & 0.264+/-0.255 -- close, correct qualitative pattern; exact '
                         'match not expected since this is a full pipeline rebuild (different splits/training run).',
    'faithfulness_method': 'Deletion-AUC computed on a FRESH traceable sample (n=50/group, seed=42+class), '
                            'since the original NB4/NB5 200-instance attributions had no saved row_id/indices. '
                            'Deletion curve = |score(x_masked) - score(x_original)| vs fraction of top-|attribution| '
                            'features masked to 0 (the standardized feature mean), fractions = linspace(0,1,11). '
                            'AUC via trapezoidal rule; higher = more faithful (unambiguous direction by construction, '
                            'addressing Reviewer 2 minor point 1 on Fig.4 axis clarity). VAE uses deterministic '
                            'z=mu scoring (not the MC-10-averaged stored score) for IG-compatible differentiability '
                            'and internal consistency between attribution computation and re-scoring; verified '
                            'corr ~0.999 against the MC-10 ground truth. Not directly comparable in magnitude '
                            'across models (IF bounded path-length vs AE/VAE unbounded NLL) -- use only within-model '
                            '(same dataset/model/method/class) ID-vs-ID2-vs-OOD comparisons.',
    'models_verified_against_stored_scores': {
        'ae_credit_card': 'exact match (mean-MSE, ratio=1.0)',
        'vae_credit_card': 'corr=0.9994 (MC sampling noise only)',
        'if_credit_card': 'exact match (diff~1e-16)',
        'ae_scenario_T': 'pearson=1.0',
        'vae_scenario_T': 'pearson=0.99994',
        'if_scenario_T': 'exact match',
        'ae_scenario_A': 'pearson=0.9999998',
        'vae_scenario_A': 'pearson=0.9998522',
        'if_scenario_A': 'exact match',
    },
}
with open('/kaggle/working/nb6_metadata.json', 'w') as f:
    json.dump(nb6_metadata, f, indent=2)

# Zip everything for download/upload as a new dataset
zip_path = '/kaggle/working/nb6_complete.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['nb6_stability_results.csv', 'nb6_faithfulness_results.csv', 'nb6_metadata.json']:
        zf.write(f'/kaggle/working/{fname}', fname)

print("Files ready in /kaggle/working:")
for f in ['nb6_stability_results.csv', 'nb6_faithfulness_results.csv', 'nb6_metadata.json', 'nb6_complete.zip']:
    p = f'/kaggle/working/{f}'
    print(f"  {f}: {os.path.getsize(p)} bytes")

print(f"\nstability_df: {len(stability_df)} rows")
print(f"faithfulness_df: {len(faithfulness_df)} rows")

Files ready in /kaggle/working:
  nb6_stability_results.csv: 5712 bytes
  nb6_faithfulness_results.csv: 7699 bytes
  nb6_metadata.json: 1956 bytes
  nb6_complete.zip: 6697 bytes

stability_df: 60 rows
faithfulness_df: 90 rows


In [25]:
import pickle

raw_stability = {}  # raw_stability[(dataset,method,model,class,comparison)] = np.array of per-pair correlations

for ds in datasets:
    for method_name, meta in methods.items():
        method_dict = meta['dict']
        for model in meta['models']:
            if model not in method_dict.get(ds, {}):
                continue
            for cls in [0, 1]:
                for split_a, split_b, comp_name in comparisons:
                    try:
                        arr_a = method_dict[ds][model][split_a][cls]
                        arr_b = method_dict[ds][model][split_b][cls]
                    except KeyError:
                        continue
                    corrs = paired_spearman_stability(arr_a, arr_b)
                    raw_stability[(ds, method_name, model, cls, comp_name)] = corrs

with open('/kaggle/working/nb6_raw_stability_arrays.pkl', 'wb') as f:
    pickle.dump(raw_stability, f)

print(f"Saved {len(raw_stability)} raw stability arrays")
print("Example:", list(raw_stability.keys())[0], "-> shape", raw_stability[list(raw_stability.keys())[0]].shape)

Saved 60 raw stability arrays
Example: ('credit_card', 'ig', 'ae', 0, 'ID_vs_OOD') -> shape (200,)


In [26]:
raw_faithfulness = {}  # raw_faithfulness[(dataset,method,model,class,split)] = np.array of per-instance AUCs

for ds in ['credit_card', 'scenario_T', 'scenario_A']:
    for model_name in ['ae', 'vae']:
        score_fn = score_fn_builders[model_name](models[ds][model_name])
        for split in ['id', 'id2', 'ood']:
            for cls in [0, 1]:
                d = faith_ig[ds][model_name][split][cls]
                if len(d['row_id']) == 0:
                    continue
                aucs = compute_deletion_auc(d['X'], d['attrs'], score_fn)
                raw_faithfulness[(ds, 'ig', model_name, cls, split)] = aucs

for ds in ['credit_card', 'scenario_T', 'scenario_A']:
    for model_name in ['ae', 'vae', 'if']:
        score_fn = score_fn_builders[model_name](models[ds][model_name])
        for split in ['id', 'id2', 'ood']:
            for cls in [0, 1]:
                d = faith_shap[ds][model_name][split][cls]
                X = faith_samples[ds][split][cls]['X'].numpy()
                sv = np.asarray(d['shap_values'])
                aucs = compute_deletion_auc(X, sv, score_fn)
                raw_faithfulness[(ds, 'shap', model_name, cls, split)] = aucs

with open('/kaggle/working/nb6_raw_faithfulness_arrays.pkl', 'wb') as f:
    pickle.dump(raw_faithfulness, f)

print(f"Saved {len(raw_faithfulness)} raw faithfulness arrays")
print("Example:", list(raw_faithfulness.keys())[0], "-> shape", raw_faithfulness[list(raw_faithfulness.keys())[0]].shape)

/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or o

Saved 90 raw faithfulness arrays
Example: ('credit_card', 'ig', 'ae', 0, 'id') -> shape (43,)


/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)
/tmp/ipykernel_58/3336966850.py:50: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(delta_matrix, fractions, axis=1)


In [27]:
zip_path = '/kaggle/working/nb6_complete.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['nb6_stability_results.csv', 'nb6_faithfulness_results.csv', 'nb6_metadata.json',
                  'nb6_raw_stability_arrays.pkl', 'nb6_raw_faithfulness_arrays.pkl']:
        zf.write(f'/kaggle/working/{fname}', fname)

print("Re-zipped nb6_complete.zip with raw arrays included:")
for f in ['nb6_stability_results.csv', 'nb6_faithfulness_results.csv', 'nb6_metadata.json',
          'nb6_raw_stability_arrays.pkl', 'nb6_raw_faithfulness_arrays.pkl', 'nb6_complete.zip']:
    p = f'/kaggle/working/{f}'
    print(f"  {f}: {os.path.getsize(p)} bytes")

Re-zipped nb6_complete.zip with raw arrays included:
  nb6_stability_results.csv: 5712 bytes
  nb6_faithfulness_results.csv: 7699 bytes
  nb6_metadata.json: 1956 bytes
  nb6_raw_stability_arrays.pkl: 93900 bytes
  nb6_raw_faithfulness_arrays.pkl: 40335 bytes
  nb6_complete.zip: 91708 bytes
